# Explorando o pipeline

Notebook de apoio à PoC. Serve para **olhar os dados em cada camada** sem sair do VS Code.

Ordem de uso:

1. rode `python -m src.pipeline` (cria o Bronze) — depois use as seções 1 a 4;
2. rode `dbt run` dentro de `dbt/` (cria Silver e Gold) — depois use as seções 5 e 6.

> Requer o kernel do ambiente do projeto (`.venv` ou `poc`). No VS Code: canto superior direito → *Select Kernel*.

## 0. Preparação

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

# a raiz do projeto (este notebook está em notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Raiz do projeto:", RAIZ)

con = duckdb.connect()          # DuckDB em memória: só para consultar arquivos
con.execute(f"set file_search_path='{RAIZ}'")

def q(sql):
    """Executa SQL e devolve um DataFrame (renderiza bonito no notebook)."""
    return con.execute(sql).df()

Raiz do projeto: /Users/vanessaborges/dev_repo/poc-residencia/data-pipeline-poc


## 1. As fontes: o que a origem nos entregou

Antes de qualquer transformação. Repare que **tudo é texto** aqui — CSV não guarda tipo.

In [2]:
q("select * from 'data/fontes_poc/processos.csv' limit 10")

,processo_id,classe_id,comarca_id,vara_id,data_distribuicao,data_baixa,situacao
0,8000001,1,4,3,2021-09-30,2022-05-27,Baixado
1,8000002,5,4,1,2023-09-25,2024-05-03,Baixado
2,8000003,5,1,5,2019-04-19,2023-01-08,Baixado
3,8000004,3,5,2,2024-01-14,2024-02-26,Baixado
4,8000005,1,5,4,2022-10-25,2024-02-07,Baixado
5,8000006,3,8,1,2020-01-15,2022-02-18,Baixado
6,8000007,6,6,3,2019-06-27,2022-07-29,Baixado
7,8000008,3,1,1,11/03/2025,2028-10-18,Baixado
8,8000009,3,3,5,2021-02-26,2021-06-29,Baixado
9,8000010,2,4,1,2021-08-11,NaN,Em andamento


In [3]:
# o JSON de movimentações
q("select * from 'data/fontes_poc/movimentacoes.json' limit 10")

,processo_id,tipo_movimento,data_movimento
0,8000097,Baixa Definitiva,2019-07-19
1,8000070,Sentença,2020-10-12
2,8000067,Juntada de Petição,2019-12-17
3,8000070,Juntada de Petição,2020-05-13
4,8000044,Despacho,2021-04-10
5,8000013,Conclusão,2025-07-23
6,8000019,Juntada de Petição,2019-11-18
7,8000026,Baixa Definitiva,2020-02-24
8,8000089,Juntada de Petição,2022-06-10
9,8000087,Juntada de Petição,2025-03-07


## 2. O Bronze: o que o pipeline capturou

Mesmo conteúdo das fontes, em Parquet, com o metadado de ingestão.

In [4]:
q("select * from 'data/bronze/processos.parquet' limit 10")

,processo_id,classe_id,comarca_id,vara_id,data_distribuicao,data_baixa,situacao,_ingerido_em
0,8000001,1,4,3,2021-09-30,2022-05-27,Baixado,2026-08-13T20:23:29.534371+00:00
1,8000002,5,4,1,2023-09-25,2024-05-03,Baixado,2026-08-13T20:23:29.534371+00:00
2,8000003,5,1,5,2019-04-19,2023-01-08,Baixado,2026-08-13T20:23:29.534371+00:00
3,8000004,3,5,2,2024-01-14,2024-02-26,Baixado,2026-08-13T20:23:29.534371+00:00
4,8000005,1,5,4,2022-10-25,2024-02-07,Baixado,2026-08-13T20:23:29.534371+00:00
5,8000006,3,8,1,2020-01-15,2022-02-18,Baixado,2026-08-13T20:23:29.534371+00:00
6,8000007,6,6,3,2019-06-27,2022-07-29,Baixado,2026-08-13T20:23:29.534371+00:00
7,8000008,3,1,1,11/03/2025,2028-10-18,Baixado,2026-08-13T20:23:29.534371+00:00
8,8000009,3,3,5,2021-02-26,2021-06-29,Baixado,2026-08-13T20:23:29.534371+00:00
9,8000010,2,4,1,2021-08-11,NaN,Em andamento,2026-08-13T20:23:29.534371+00:00


### 2.1 Por que Parquet? Olhe o arquivo por dentro

O `parquet_metadata` mostra **uma linha por coluna** — é a prova de que o formato é colunar.
Compare os tamanhos: colunas com poucos valores distintos (`situacao`) comprimem muito
mais que colunas com valores únicos (`processo_id`).

In [ ]:
q("""
    select
        path_in_schema            as coluna,
        compression               as compressao,
        total_uncompressed_size   as bytes_sem_compressao,
        total_compressed_size     as bytes_comprimidos
    from parquet_metadata('data/bronze/processos.parquet')
""")

In [ ]:
# tamanho em disco: CSV x Parquet
csv = (RAIZ / "data/fontes_poc/processos.csv").stat().st_size
pq  = (RAIZ / "data/bronze/processos.parquet").stat().st_size
print(f"CSV:     {csv:>7,} bytes")
print(f"Parquet: {pq:>7,} bytes")
print(f"\nObs.: com 122 linhas o Parquet pode até ser MAIOR — ele carrega schema e")
print("estatísticas no cabeçalho. A vantagem aparece na escala (milhões de linhas)")
print("e na leitura seletiva de colunas.")

## 3. Caça aos problemas de qualidade

O Bronze preserva o que chegou — **inclusive os defeitos**. Encontre-os aqui
antes de decidir como tratá-los na Silver.

In [ ]:
# a) datas em formatos diferentes?
q("""
    select data_distribuicao, count(*) as n
    from 'data/bronze/processos.parquet'
    where data_distribuicao like '%/%'
    group by 1
""")

In [ ]:
# b) registros duplicados?
q("""
    select processo_id, count(*) as vezes
    from 'data/bronze/processos.parquet'
    group by 1
    having count(*) > 1
""")

In [ ]:
# c) identificadores ausentes?
q("""
    select *
    from 'data/bronze/processos.parquet'
    where processo_id is null or processo_id = ''
""")

In [ ]:
# d) nomes de comarca padronizados?
q("select comarca_id, nome_comarca from 'data/bronze/comarcas.parquet' order by comarca_id")

In [ ]:
# e) classes: o mesmo id aparece mais de uma vez?
q("select * from 'data/bronze/classes.parquet' order by classe_id")

In [ ]:
# f) integridade: existe processo apontando para comarca inexistente?
q("""
    select p.processo_id, p.comarca_id
    from 'data/bronze/processos.parquet' p
    left join 'data/bronze/comarcas.parquet' c
           on p.comarca_id = c.comarca_id
    where c.comarca_id is null
""")

## 4. Um resumo do que você encontrou

Anote aqui (em texto mesmo) os problemas e a decisão que pretende tomar para cada um.
Essa lista vira o roteiro dos TODOs da Silver.

| # | Problema | Onde | Decisão |
|---|---|---|---|
| 1 | | | |
| 2 | | | |

## 5. Depois do `dbt run`: Silver e Gold

Execute dentro da pasta `dbt/`:

```bash
dbt run
```

E então compare as camadas.

In [ ]:
# Silver: dado tipado e limpo (repare nos TIPOS das colunas!)
q("select * from 'data/silver/stg_processos.parquet' limit 10")

In [ ]:
# os tipos mudaram do Bronze para a Silver?
bronze = q("select name, type from parquet_schema('data/bronze/processos.parquet') where name <> 'schema'")
silver = q("select name, type from parquet_schema('data/silver/stg_processos.parquet') where name <> 'schema'")
bronze.merge(silver, on="name", how="outer", suffixes=("_bronze", "_silver"))

In [ ]:
# Gold: o modelo dimensional
q("select * from 'data/gold/fato_processo.parquet' limit 10")

## 6. A pergunta de negócio

> Qual é o tempo médio de tramitação por comarca, classe e período?

Só funciona depois que a medida `tempo_tramitacao_dias` estiver implementada na `fato_processo`.

In [ ]:
q("""
    select
        c.nome_comarca,
        cl.nome_classe,
        t.ano,
        count(*)                               as processos,
        round(avg(f.tempo_tramitacao_dias), 1) as tempo_medio_dias
    from 'data/gold/fato_processo.parquet' f
    join 'data/gold/dim_comarca.parquet'  c  on f.comarca_id = c.comarca_id
    join 'data/gold/dim_classe.parquet'   cl on f.classe_id  = cl.classe_id
    join 'data/gold/dim_tempo.parquet'    t  on f.data_distribuicao = t.data
    where f.tempo_tramitacao_dias is not null
    group by 1, 2, 3
    order by 1, 2, 3
""")

### 6.1 Variações — e uma armadilha

O gestor nunca pede só uma coisa. Abaixo: por comarca e ano, por vara — e o erro que é fácil cometer.

In [ ]:
# por comarca e ano: está melhorando ou piorando?
q("""
    select
        c.nome_comarca,
        t.ano,
        count(*)                                 as processos_baixados,
        round(avg(f.tempo_tramitacao_dias), 1)   as tempo_medio_dias
    from 'data/gold/fato_processo.parquet' f
    join 'data/gold/dim_comarca.parquet' c on f.comarca_id = c.comarca_id
    join 'data/gold/dim_tempo.parquet'   t on f.data_distribuicao = t.data
    where f.tempo_tramitacao_dias is not null
    group by 1, 2
    order by 1, 2
""")

In [ ]:
# ⚠ A ARMADILHA: vara_id é único no estado?
q("""
    select
        vara_id,
        count(*)                    as processos,
        count(distinct comarca_id)  as comarcas_diferentes
    from 'data/gold/fato_processo.parquet'
    where tempo_tramitacao_dias is not null
    group by 1 order by 1
""")
# Olhe a última coluna: a mesma "vara 2" aparece em várias comarcas.
# Agrupar só por vara_id somaria a 2ª Vara de Campo Grande com a de Dourados.

In [ ]:
# por vara, do jeito certo: SEMPRE com a comarca + significância mínima
q("""
    select
        c.nome_comarca,
        f.vara_id                                as vara,
        count(*)                                 as processos_baixados,
        round(avg(f.tempo_tramitacao_dias), 1)   as tempo_medio_dias
    from 'data/gold/fato_processo.parquet' f
    join 'data/gold/dim_comarca.parquet' c on f.comarca_id = c.comarca_id
    where f.tempo_tramitacao_dias is not null
    group by 1, 2
    having count(*) >= 3
    order by tempo_medio_dias desc
""")

---

**Fechando o raciocínio:** o mesmo dado apareceu quatro vezes neste notebook —
na fonte, no Bronze, na Silver e na Gold. O que mudou a cada passo?

- fonte → Bronze: o **formato**
- Bronze → Silver: o **conteúdo** (qualidade)
- Silver → Gold: a **organização** (modelo)
- Gold → consulta: o **significado** (informação para decidir)